In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

In [0]:
# Change the catalog and schema here if necessary.
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"

In [0]:
# Read the churn dataset (adjust the table name if yours differs)
import pandas as pd
from sklearn.model_selection import train_test_split

TABLE_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.telco_customer_churn_silver"
df = spark.read.table(TABLE_NAME).toPandas()

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
display(df.head())

# Target: "Churn" (Yes/No) -> binary classification
TARGET_COL = "Churn"
print("Target distribution:")
print(df[TARGET_COL].value_counts(normalize=True))

# --- Light preprocessing ---
# Drop the identifier column (no predictive value)
df = df.drop(columns=["customerID"], errors="ignore")

# Telco churn quirk: TotalCharges is stored as a string with blanks for new customers
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Feature/type summary (feeds the ColumnTransformer in the next step)
feature_types = {c: ("numeric" if pd.api.types.is_numeric_dtype(df[c]) else "categorical")
                for c in df.columns if c != TARGET_COL}
print(feature_types)

# Keep raw feature values and string labels: encoding/imputation/scaling will live
# INSIDE the sklearn Pipeline we fit in the next cell (avoids train/serve skew).
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# No event/observation timestamp -> rows are exchangeable: stratified random split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Churn rate (train): {y_train.eq('Yes').mean():.3f}, (test): {y_test.eq('Yes').mean():.3f}")

In [0]:
# Build preprocessing + model pipeline
# Categorical features are one-hot encoded; numeric features are scaled.
# Everything lives inside the Pipeline so the same transforms apply at inference.
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import mlflow
from mlflow.models import infer_signature

numeric_cols = [c for c, t in feature_types.items() if t == "numeric"]
cat_cols = [c for c, t in feature_types.items() if t == "categorical"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", cat_transformer, cat_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
])

mlflow.set_experiment("/Shared/churn_rf_experiment")

with mlflow.start_run(run_name="rf_baseline") as run:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("accuracy", (y_pred == y_test).mean())

    signature = infer_signature(X_train.head(100), model.predict(X_train.head(100)))
    model_info = mlflow.sklearn.log_model(
        model, name="churn_rf_model", signature=signature, input_example=X_train.head(3)
    )

    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {auc:.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nMLflow run: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")

In [0]:
# Register the RandomForest model in Unity Catalog
# Reuses the model_info from the training cell (no re-logging needed).
import mlflow

mlflow.set_registry_uri("databricks-uc")

registered_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.churn_rf_model"

# --- Validate the artifact before registration ---
# Confirms the model has a signature, an input example, and can predict in a clean environment.
pyfunc_model = mlflow.pyfunc.load_model(model_info.model_uri)
assert pyfunc_model.metadata.signature is not None, "Model has no signature — fix the original artifact."
assert pyfunc_model.input_example is not None, "Model has no input example — fix the original artifact."

mlflow.models.predict(
    model_uri=model_info.model_uri,
    input_data=pyfunc_model.input_example,
    env_manager="uv",
)
print("✓ Artifact validated (signature, input example, and clean-env predict all passed).")

# --- Register in Unity Catalog ---
registered_version = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=registered_model_name,
    await_registration_for=300,
)
model_version = registered_version.version

print(f"✓ Registered: {registered_model_name} (version {model_version})")
print(f"  URI: models:/{registered_model_name}/{model_version}")